# Task 4 — Parte A: otimização de hiperparâmetros

Este notebook faz a **primeira metade da Task 4**: escolher os modelos mais promissores da Task 3, otimizar os seus hiperparâmetros e perceber se a otimização traz uma melhoria **real e estável**. No fim, guarda o que a Parte B (limiar, teste e interpretação) precisa.

**Como funciona**

- Usa o `avaliar_folds` e a restante infraestrutura da Task 2 (`avaliacao.py`) e o pré-processamento da Task 3 (`ablacao.py`): 4 folds temporais, recall com 200 alertas por 1000 sessões como métrica principal. O pré-processamento e o modelo são **reajustados em cada fold, só com o treino desse fold**. O teste (out–dez/2021) é removido ao carregar os dados e nunca é usado aqui.
- As funções de pesquisa e de escolha estão em `otimizacao.py`. O notebook só chama essas funções e mostra os resultados.
- **Os resultados vão sendo guardados** em `Resultados/` durante a pesquisa. Se o computador adormecer ou o kernel falhar, volta-se a correr a mesma célula e só se calcula o que falta.

| Secção | O que responde |
|---|---|
| 0 | Configuração e dados |
| 1 | Que modelos otimizar, e porquê |
| 2 | Procedimento de pesquisa e espaços de pesquisa |
| 3 | As três pesquisas (logística, boosting, Random Forest) |
| 4 | Inicial contra otimizado: média, variabilidade e evolução no tempo |
| 5 | Verificação sem otimismo: escolher com F1–F3 e avaliar em F4 |
| 6 | Qual é o modelo probabilístico mais forte |
| 7 | Ficheiros para a Parte B |
| 8 | Conclusões: o que escrever no relatório |

## 0. Configuração e dados

In [1]:
import json
import warnings
from pathlib import Path
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"   # põe o número de núcleos do teu computador
import numpy as np
import pandas as pd

import ablacao as ab
import otimizacao as ot
from avaliacao import avaliar_folds, comparar

warnings.filterwarnings("ignore", category=FutureWarning)   # os avisos de convergência continuam visíveis
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 90)

DATA_PATH = Path("Charging_Data_educational.csv")
PASTA = Path("Resultados")
PASTA.mkdir(exist_ok=True)

X_ALERTAS = 200      # orçamento de alertas por 1000 sessões (secção 8.4 do relatório)
SEED = 0             # semente da pesquisa aleatória
AMOSTRA = 1          # 1 = todos os dados. Para um ensaio rápido, 8 (volta a 1 para os resultados finais)
ESPACO = None        # sem informação espacial: com o histórico, o posto não acrescenta nada (secção 9.1 e 9.2)
GRUPOS_TODOS = ab.CONFIGURACOES["4 + histórico do utilizador"]
MODELOS = ["logistica", "boosting", "random_forest"]
NOMES = {"logistica": "Regressão logística", "boosting": "Boosting", "random_forest": "Random Forest"}

In [2]:
df = ab.carregar_dev(DATA_PATH)            # só treino e validação (dez/2019 – set/2021)
assert df["Start Time"].max() < pd.Timestamp("2021-10-01"), "o teste não pode estar carregado"
if AMOSTRA > 1:
    df = df.iloc[::AMOSTRA].reset_index(drop=True)
    print(f"ATENÇÃO: ensaio rápido com 1 em cada {AMOSTRA} sessões. Não usar estes resultados.")
print(f"{len(df):,} sessões | {df['Start Time'].min():%Y-%m-%d} a {df['Start Time'].max():%Y-%m-%d} | prevalência {df['is_Abnormal'].mean():.1%}")


def metricas_fold(res):
    return pd.Series({"recall@200 (média)": res["recall"].mean(), "desvio": res["recall"].std(ddof=1),
                      "pior fold": res["recall"].min(), "PR-AUC norm. (média)": res["pr_auc_norm"].mean(),
                      "ROC-AUC (média)": res["roc_auc"].mean()})

360,996 sessões | 2019-12-31 a 2021-09-30 | prevalência 18.8%


## 1. Que modelos otimizar, e porquê

O enunciado pede **dois ou três** modelos promissores. A escolha apoia-se nos resultados da Task 3 (combinação 4, recall com 200 alertas por 1000 sessões, média dos 4 folds):

| Modelo (Task 3) | Recall | Pior fold | PR-AUC norm. | Tempo (4 folds) | Decisão |
|---|---|---|---|---|---|
| Random Forest | 0,588 | 0,576 | 0,547 | 255 s | **Otimizar.** Melhor recall e melhor PR-AUC; é o modelo preferido pela regra da secção 8.9. |
| Boosting | 0,586 | 0,572 | 0,544 | 23 s | **Otimizar.** Quase empata com a Random Forest, custa dez vezes menos e tem mais hiperparâmetros com efeito (profundidade, folhas, regularização), logo mais margem para melhorar. |
| Regressão logística | 0,579 ¹ | 0,566 | 0,521 | 21 s | **Otimizar.** É o melhor modelo interpretável e a referência probabilística: mostra se os modelos complexos justificam o custo. Já se viu na Task 3 que o `C` não altera o recall, por isso é um teste da hipótese "o modelo está limitado pela informação, não pela configuração". |
| Árvore de decisão | 0,578 | 0,564 | 0,525 | 24 s | Não. É dominada pela Random Forest, que é um conjunto destas árvores. |
| Naive Bayes | 0,574 | 0,559 | 0,498 | 2 s | Não. É o mais fraco e as probabilidades são demasiado extremas (a independência não se cumpre). |
| Votação (logística + árvore) | 0,583 | 0,572 | 0,536 | 45 s | Não. Combina modelos já afinados: otimizá-la seria otimizar os seus componentes. |

¹ Sem informação espacial, como nas pesquisas desta Task. Com o posto, a logística tem 0,580 (secção 9.2); a diferença é inferior a 0,001.

**Expectativa.** Na Task 3 as diferenças entre modelos foram inferiores a 0,015 e a afinação simples trouxe, no máximo, +0,003. É provável que a otimização acrescente pouco. Concluir que **não trouxe uma melhoria real** é um resultado válido, desde que esteja demonstrado.

## 2. Procedimento de pesquisa

| Elemento | Decisão |
|---|---|
| Dados | Só treino e validação (dez/2019 a set/2021). O teste não é carregado. |
| Ordem temporal | Os mesmos 4 folds de janela crescente da Task 2 (validações trimestrais de out/2020 a set/2021). |
| Pré-processamento | Reajustado em cada fold com o treino desse fold (`fabricar_scorer`). Nada é calculado com a validação. |
| Variáveis | Combinação 4, sem informação espacial (como a Random Forest e o boosting na Task 3). |
| Método | **Pesquisa aleatória** com semente fixa (`SEED`). Há cinco a sete hiperparâmetros por modelo, e a pesquisa aleatória cobre melhor o espaço do que uma grelha com o mesmo número de corridas. Na logística, com dois hiperparâmetros, usa-se a **grelha** completa. |
| Métrica a otimizar | **Recall com 200 alertas por 1000 sessões**, a métrica principal do projeto (secção 8.4), com a regra da secção 8.9: diferenças inferiores a 0,005 são empate, **desempatado pela PR-AUC normalizada**.|
| Comparação | A configuração escolhida é comparada com a **inicial** (a que saiu da Task 3), fold a fold, com a regra completa da secção 8.9: é preferida se tiver maior média, for melhor em pelo menos 3 de 4 folds e tiver pior fold não inferior. Se falhar uma destas condições, é empate, e prefere-se a de menor desvio-padrão entre folds, depois a de maior PR-AUC normalizada e por fim a mais simples. A mesma regra escolhe o modelo probabilístico mais forte (secção 6). |
| Poupar tempo | Na Random Forest usam-se 100 árvores durante a pesquisa e 200 na avaliação final. A configuração inicial é avaliada nas mesmas condições, por isso a comparação é justa. |

**Porque o recall e não a PR-AUC.** O recall com 200 alertas é o critério de decisão do projeto, mas num fold varia por acaso cerca de 0,004 a 0,006 (secção 8.9). Por isso diferenças inferiores a 0,005 contam como empate e a PR-AUC normalizada, que usa todas as previsões, desempata.

**Cuidados**

- Os mesmos folds servem para **escolher** e para **comparar**, por isso o ganho medido é **ligeiramente otimista**. A secção 5 faz uma verificação sem esse otimismo: escolhe-se só com F1–F3 e avalia-se em F4.
- Os resultados ficam em `Resultados/pesquisa_<modelo>_amostra<N>.csv`. Apaga o ficheiro se alterares o espaço de pesquisa.

In [3]:
# Configuração de partida (a que saiu da Task 3; o que não é indicado vem de ab.HIPERPARAMETROS)
INICIAL = {
    "logistica":     {"C": 0.1, "class_weight": None},
    "boosting":      {"learning_rate": 0.05, "max_iter": 200},
    "random_forest": {"min_samples_leaf": 50, "max_features": 0.5},
}

# Espaços de pesquisa
ESPACOS = {
    "logistica": {"C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10],
                  "class_weight": [None, "balanced"]},
    "boosting": {"learning_rate": [0.02, 0.03, 0.05, 0.08, 0.1],
                 "max_iter": [100, 150, 200, 300, 400],
                 "max_leaf_nodes": [7, 15, 31, 63],
                 "max_depth": [None, 3, 5, 8],
                 "min_samples_leaf": [20, 50, 100, 200, 500],
                 "l2_regularization": [0, 0.1, 1, 10],
                 "max_features": [0.5, 0.8, 1.0]},
    "random_forest": {"min_samples_leaf": [10, 25, 50, 100],
                      "max_features": ["sqrt", 0.3, 0.5, 0.7],
                      "max_depth": [None, 12, 20],
                      "max_samples": [0.3, 0.5, 0.7, None]},
}

# Número de configurações por modelo (a grelha da logística tem 16 pontos)
N_CONFIG = {"logistica": 16, "boosting": 40, "random_forest": 16}
# Só durante a pesquisa: menos árvores na Random Forest, para poupar tempo
FIXOS_PESQUISA = {"random_forest": {"n_estimators": 100}}

pd.DataFrame({NOMES[m]: {"configurações testadas": N_CONFIG[m],
                         "pontos no espaço": int(np.prod([len(v) for v in ESPACOS[m].values()])),
                         "hiperparâmetros": ", ".join(ESPACOS[m])} for m in MODELOS}).T

,configurações testadas,pontos no espaço,hiperparâmetros
Regressão logística,16,16,"C, class_weight"
Boosting,40,24000,"learning_rate, max_iter, max_leaf_nodes, max_depth, min_samples_leaf, l2_regularizatio..."
Random Forest,16,192,"min_samples_leaf, max_features, max_depth, max_samples"


**Porque estes intervalos**

- **Logística.** `C` (inverso da regularização) de 0,003 a 10, em escala logarítmica, que cobre desde muito regularizada até quase sem regularização. `class_weight` testa se dar mais peso às falhas ajuda; na Task 3 não ajudou com o `C` fixo.
- **Boosting.** Taxa de aprendizagem e número de iterações (andam juntas: taxa baixa pede mais iterações); `max_leaf_nodes` e `max_depth` controlam a complexidade de cada árvore; `min_samples_leaf` e `l2_regularization` limitam o sobreajuste; `max_features` sorteia uma fração das variáveis em cada divisão. O valor inicial (taxa 0,05, 200 iterações, 31 folhas, folha mínima 100) está dentro do espaço.
- **Random Forest.** `min_samples_leaf` e `max_depth` controlam o tamanho das árvores; `max_features` e `max_samples` (tamanho da amostra bootstrap) controlam a diversidade entre árvores, que é o que faz a floresta melhorar uma árvore isolada.

## 3. As três pesquisas

In [4]:
def mostrar(tabela, n=10):
    """Tabela das melhores configurações (a inicial vem marcada com *)."""
    t = tabela.sort_values("recall_media", ascending=False).head(n).copy()
    t["configuração"] = np.where(t["inicial"], "* ", "") + t["hp"]
    return t[["configuração", "recall_media", "recall_desvio", "recall_pior", "prauc_media", "roc_media", "tempo_s"]] \
        .rename(columns={"recall_media": "recall@200 (média)", "recall_desvio": "desvio", "recall_pior": "pior fold",
                         "prauc_media": "PR-AUC norm.", "roc_media": "ROC-AUC", "tempo_s": "tempo (s)"}).round(4)

tabelas = {}


def correr_pesquisa(m):
    lista = ot.candidatos(ESPACOS[m], N_CONFIG[m], seed=SEED)
    tabelas[m] = ot.pesquisar(df, m, lista, GRUPOS_TODOS, ESPACO, X_ALERTAS, fixos=FIXOS_PESQUISA.get(m),
                              inicial=INICIAL[m], ficheiro=PASTA / f"pesquisa_{m}_amostra{AMOSTRA}.csv")
    return mostrar(tabelas[m])

### 3.1 Regressão logística (grelha)

In [5]:
correr_pesquisa("logistica")

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
1,"{""C"": 0.003, ""class_weight"": null}",0.5793,0.0142,0.5657,0.5190,0.8297,6.4388
3,"{""C"": 0.01, ""class_weight"": null}",0.5791,0.0138,0.5669,0.5198,0.8298,7.6765
5,"{""C"": 0.03, ""class_weight"": null}",0.5790,0.0139,0.5665,0.5200,0.8298,7.9790
10,"{""C"": 1, ""class_weight"": null}",0.5790,0.0136,0.5663,0.5202,0.8297,6.1211
8,"{""C"": 0.3, ""class_weight"": null}",0.5790,0.0136,0.5663,0.5202,0.8297,6.0263
14,"{""C"": 10, ""class_weight"": null}",0.5789,0.0136,0.5663,0.5202,0.8297,5.8716
12,"{""C"": 3, ""class_weight"": null}",0.5789,0.0136,0.5663,0.5202,0.8297,5.8975
0,"* {""C"": 0.1, ""class_weight"": null}",0.5788,0.0136,0.5664,0.5202,0.8298,6.1865
6,"{""C"": 0.03, ""class_weight"": ""balanced""}",0.5784,0.0144,0.5643,0.5165,0.8298,6.2661
9,"{""C"": 0.3, ""class_weight"": ""balanced""}",0.5784,0.0144,0.5645,0.5166,0.8298,6.7471


Conclusão. As 16 configurações ficam entre 0,578 e 0,579 de recall, uma diferença inferior a 0,005, por isso empatam todas. O desempate pela PR-AUC normalizada escolhe C = 3, mas a diferença para a inicial (C = 0,1) só aparece na quarta casa decimal. class_weight="balanced" é ligeiramente pior. A comparação com a inicial é feita na secção 4.

### 3.2 Boosting (pesquisa aleatória, 40 configurações)

In [6]:
correr_pesquisa("boosting")

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
25,"{""l2_regularization"": 0.1, ""learning_rate"": 0.02, ""max_depth"": null, ""max_features"": 0...",0.5882,0.0132,0.5750,0.5469,0.8359,28.6916
10,"{""l2_regularization"": 1, ""learning_rate"": 0.03, ""max_depth"": 8, ""max_features"": 0.5, ""...",0.5877,0.0136,0.5746,0.5461,0.8357,15.7102
37,"{""l2_regularization"": 0, ""learning_rate"": 0.02, ""max_depth"": 5, ""max_features"": 0.8, ""...",0.5868,0.0135,0.5732,0.5445,0.8355,44.6830
29,"{""l2_regularization"": 0.1, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5,...",0.5867,0.0137,0.5734,0.5445,0.8354,22.4784
31,"{""l2_regularization"": 0, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5, ""...",0.5867,0.0134,0.5735,0.5444,0.8355,24.9761
9,"{""l2_regularization"": 10, ""learning_rate"": 0.08, ""max_depth"": 5, ""max_features"": 0.5, ...",0.5867,0.0131,0.5742,0.5445,0.8353,13.9899
15,"{""l2_regularization"": 10, ""learning_rate"": 0.03, ""max_depth"": 5, ""max_features"": 0.5, ...",0.5865,0.0130,0.5742,0.5445,0.8354,21.6418
13,"{""l2_regularization"": 1, ""learning_rate"": 0.08, ""max_depth"": null, ""max_features"": 1.0...",0.5864,0.0137,0.5734,0.5437,0.8349,14.2920
0,"* {""learning_rate"": 0.05, ""max_iter"": 200}",0.5862,0.0141,0.5730,0.5443,0.8349,18.8015
28,"{""l2_regularization"": 0, ""learning_rate"": 0.1, ""max_depth"": null, ""max_features"": 0.5,...",0.5859,0.0130,0.5745,0.5416,0.8339,25.9638


Conclusão. As melhores configurações têm taxa de aprendizagem baixa (0,02 a 0,03). A escolhida (taxa 0,02, 150 iterações, até 63 folhas, folha mínima de 50, metade das variáveis por divisão, L2 de 0,1, profundidade ilimitada) tem o maior recall (0,5882) e a maior PR-AUC normalizada (0,5469) da pesquisa, contra 0,5862 e 0,5443 da inicial. A diferença de recall (+0,002) é inferior a 0,005; a secção 4 verifica se é estável.

### 3.3 Random Forest (pesquisa aleatória, 16 configurações)

In [7]:
correr_pesquisa("random_forest")

,configuração,recall@200 (média),desvio,pior fold,PR-AUC norm.,ROC-AUC,tempo (s)
7,"{""max_depth"": null, ""max_features"": 0.3, ""max_samples"": 0.3, ""min_samples_leaf"": 50}",0.5882,0.0134,0.5755,0.5452,0.8347,41.9620
9,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": 0.3, ""min_samples_leaf"": 25}",0.5881,0.0129,0.5769,0.5451,0.8341,46.0543
2,"{""max_depth"": 12, ""max_features"": ""sqrt"", ""max_samples"": 0.7, ""min_samples_leaf"": 50}",0.5880,0.0138,0.5756,0.5443,0.8350,51.3054
8,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": null, ""min_samples_leaf"": 25}",0.5879,0.0122,0.5767,0.5460,0.8338,112.3071
13,"{""max_depth"": 20, ""max_features"": 0.3, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5878,0.0129,0.5764,0.5466,0.8347,108.6379
0,"* {""max_features"": 0.5, ""min_samples_leaf"": 50}",0.5878,0.0133,0.5762,0.5462,0.8341,204.5700
12,"{""max_depth"": 20, ""max_features"": ""sqrt"", ""max_samples"": 0.7, ""min_samples_leaf"": 100}",0.5877,0.0130,0.5757,0.5422,0.8347,48.1032
14,"{""max_depth"": 20, ""max_features"": 0.5, ""max_samples"": 0.7, ""min_samples_leaf"": 50}",0.5875,0.0129,0.5759,0.5465,0.8344,144.5230
5,"{""max_depth"": 12, ""max_features"": 0.7, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5871,0.0141,0.5746,0.5470,0.8351,201.9652
1,"{""max_depth"": 12, ""max_features"": 0.5, ""max_samples"": null, ""min_samples_leaf"": 50}",0.5870,0.0136,0.5739,0.5474,0.8354,150.8350


Conclusão. As 16 configurações ficam entre 0,586 e 0,588 de recall e empatam todas. O desempate pela PR-AUC normalizada escolhe a profundidade máxima de 12 (0,5474, contra 0,5462 da inicial), com recall ligeiramente inferior (0,5870 contra 0,5878). Há configurações com o mesmo recall da inicial em cerca de 42 s, contra cerca de 205 s. É uma vantagem de custo, não de desempenho.

## 4. Inicial contra otimizado

Para cada modelo, a configuração escolhida pela regra de hiperparâmetros da secção 8.9 (a de maior PR-AUC normalizada entre as que ficam a menos de 0,005 do melhor recall) é **avaliada de novo** nas condições finais (Random Forest com 200 árvores), tal como a inicial. Depois, a **regra completa da secção 8.9** decide qual das duas segue para a Parte B. Também se confirma que a versão escolhida continua elegível em todos os folds.

Nesta avaliação guardam-se também os **scores de aquecimento**: os 7 dias antes de cada validação, previstos pelo modelo desse fold. A Parte B precisa deles para calcular o limiar móvel da primeira semana de cada fold (secção 8.8).

In [8]:
escolhida, final, aquec = {}, {}, {}   # final[m][versao] = (res, scores); aquec[m][versao] = lista por fold
for m in MODELOS:
    linha = ot.escolher(tabelas[m])
    escolhida[m] = json.loads(linha["hp"])
    e_a_inicial = bool(linha["inicial"])
    final[m], aquec[m] = {}, {}
    for versao_, hp in [("inicial", INICIAL[m]), ("otimizado", escolhida[m])]:
        if versao_ == "otimizado" and e_a_inicial:          # a escolhida é a própria inicial: não se treina outra vez
            final[m][versao_], aquec[m][versao_] = final[m]["inicial"], aquec[m]["inicial"]
            continue
        f, guardados = ot.com_aquecimento(ab.fabricar_scorer(m, GRUPOS_TODOS, ESPACO, **hp))
        final[m][versao_] = avaliar_folds(df, f, X_ALERTAS, devolver_scores=True)
        aquec[m][versao_] = guardados
    print(f"{NOMES[m]:<20} inicial   {INICIAL[m]}\n{'':<20} escolhida {escolhida[m]}"
          + ("  (é a inicial)" if e_a_inicial else ""))

Regressão logística  inicial   {'C': 0.1, 'class_weight': None}
                     escolhida {'C': 3, 'class_weight': None}
Boosting             inicial   {'learning_rate': 0.05, 'max_iter': 200}
                     escolhida {'l2_regularization': 0.1, 'learning_rate': 0.02, 'max_depth': None, 'max_features': 0.5, 'max_iter': 150, 'max_leaf_nodes': 63, 'min_samples_leaf': 50}
Random Forest        inicial   {'min_samples_leaf': 50, 'max_features': 0.5}
                     escolhida {'max_depth': 12, 'max_features': 0.5, 'max_samples': None, 'min_samples_leaf': 50}


In [9]:
linhas = {}
for m in MODELOS:
    linhas[(NOMES[m], "inicial")] = metricas_fold(final[m]["inicial"][0])
    linhas[(NOMES[m], "otimizado")] = metricas_fold(final[m]["otimizado"][0])
pd.DataFrame(linhas).T.round(4)

recall@200 (média)  desvio  pior fold  PR-AUC norm. (média)  ROC-AUC (média)
Regressão logística inicial                0.5788  0.0136     0.5664                0.5202           0.8298
                    otimizado              0.5789  0.0136     0.5663                0.5202           0.8297
Boosting            inicial                0.5862  0.0144     0.5719                0.5441           0.8350
                    otimizado              0.5877  0.0140     0.5744                0.5466           0.8358
Random Forest       inicial                0.5877  0.0138     0.5759                0.5467           0.8344
                    otimizado              0.5877  0.0137     0.5753                0.5477           0.8356

In [10]:
# Regra da secção 8.9: o otimizado (A) contra o inicial (B), fold a fold
pd.DataFrame({NOMES[m]: ot.veredicto(final[m]["otimizado"][0], final[m]["inicial"][0]) for m in MODELOS}).T.round(4)

,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B,melhoria real e estável (regra 8.9)
Regressão logística,0.000117,0.000181,3,0.566309,0.566434,False
Boosting,0.00144,0.001361,3,0.574426,0.571866,True
Random Forest,-0.000028,0.000533,1,0.575253,0.575858,False


In [19]:
# Que versão segue para a Parte B: regra completa da secção 8.9 (com os desempates)
versao, config_final, linhas = {}, {}, {}
for m in MODELOS:
    v, tab = ot.decidir({"otimizado": final[m]["otimizado"][0], "inicial": final[m]["inicial"][0]},
                        simplicidade=["inicial", "otimizado"])
    versao[m] = v
    config_final[m] = INICIAL[m] if v == "inicial" else escolhida[m]
    linhas[NOMES[m]] = {"segue para a Parte B": v, "configuração": json.dumps(config_final[m]),
                        "comparação (A contra B)": tab.index[0], **tab.iloc[0].to_dict()}
pd.DataFrame(linhas).T

,segue para a Parte B,configuração,comparação (A contra B),diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B,melhoria real e estável (regra 8.9)
Regressão logística,inicial,"{""C"": 0.1, ""class_weight"": null}",otimizado contra inicial,0.000117,0.000181,3,0.566309,0.566434,False
Boosting,otimizado,"{""l2_regularization"": 0.1, ""learning_rate"": 0.02, ""max_depth"": null, ""max_features"": 0...",otimizado contra inicial,0.00144,0.001361,3,0.574426,0.571866,True
Random Forest,inicial,"{""min_samples_leaf"": 50, ""max_features"": 0.5}",inicial contra otimizado,0.000028,0.000533,3,0.575858,0.575253,True


In [20]:
# Elegibilidade (secção 8.9): supera, em cada fold, as duas referências da secção 8.7
REF_RECALL = {"F1": 0.520, "F2": 0.482, "F3": 0.526, "F4": 0.518}   # recall@200 da última sessão anómala
REF_PRAUC  = {"F1": 0.398, "F2": 0.354, "F3": 0.446, "F4": 0.417}   # PR-AUC norm. da taxa do posto a 7 dias
eleg = {}
for m in MODELOS:
    t = ot.elegivel(final[m][versao[m]][0], REF_RECALL, REF_PRAUC)
    eleg[NOMES[m]] = {**{f"recall {f}": t.loc[f, "recall"] for f in t.index},
                      **{f"PR-AUC {f}": t.loc[f, "PR-AUC norm."] for f in t.index},
                      "elegível em todos os folds": bool(t["supera as duas"].all())}
pd.DataFrame(eleg).T.round(4)

,recall F1,recall F2,recall F3,recall F4,PR-AUC F1,PR-AUC F2,PR-AUC F3,PR-AUC F4,elegível em todos os folds
Regressão logística,0.588571,0.567841,0.592448,0.566434,0.522492,0.462403,0.545143,0.550632,True
Boosting,0.596931,0.577069,0.602269,0.574426,0.541917,0.493239,0.572058,0.579077,True
Random Forest,0.597423,0.575858,0.601602,0.575862,0.542027,0.493345,0.570457,0.580947,True


In [21]:
# Evolução temporal: recall por fold, inicial e otimizado, e a diferença
por_fold = {}
for m in MODELOS:
    i, o = final[m]["inicial"][0]["recall"], final[m]["otimizado"][0]["recall"]
    por_fold[(NOMES[m], "inicial")] = i
    por_fold[(NOMES[m], "otimizado")] = o
    por_fold[(NOMES[m], "diferença")] = o - i
pd.DataFrame(por_fold).T.round(4)

F1      F2      F3      F4
Regressão logística inicial    0.5886  0.5678  0.5924  0.5664
                    otimizado  0.5887  0.5681  0.5926  0.5663
                    diferença  0.0001  0.0003  0.0002 -0.0001
Boosting            inicial    0.5973  0.5759  0.5999  0.5719
                    otimizado  0.5969  0.5771  0.6023  0.5744
                    diferença -0.0004  0.0012  0.0024  0.0026
Random Forest       inicial    0.5974  0.5759  0.6016  0.5759
                    otimizado  0.5973  0.5753  0.6015  0.5765
                    diferença -0.0001 -0.0006 -0.0001  0.0007

Conclusão. Nas condições finais, só o boosting otimizado é preferido à configuração inicial pela regra da secção 8.9. Tem +0,0014 de recall médio, é melhor em 3 de 4 folds (só perde no F1, −0,0004) e tem melhor pior fold (0,5744 contra 0,5719). Na logística (C = 3), a média sobe 0,0001 mas o pior fold desce ligeiramente: é empate, e os desempates mantêm a inicial. Na Random Forest, a profundidade 12 tem a mesma média (−0,00003) e só é melhor num fold, por isso a inicial é preferida. As três versões escolhidas são elegíveis em todos os folds. O ganho do boosting é estável no sentido da regra, mas muito pequeno: cerca de 0,25 falhas detetadas a mais por 1000 sessões, abaixo do erro de amostragem do recall num fold (0,004 a 0,006).

## 5. Verificação sem otimismo: escolher com F1–F3 e avaliar em F4

Os mesmos folds servem para escolher e comparar, o que favorece o modelo escolhido. Para medir isso, repete-se a escolha usando **só os folds F1–F3** (sem olhar para F4) e vê-se quanto vale essa configuração em F4, contra a inicial em F4. Não custa tempo, porque usa as tabelas já calculadas na secção 3 (mesmas condições de pesquisa).

In [22]:
linhas = {}
for m in MODELOS:
    t = tabelas[m]
    e3 = ot.escolher(t, folds=["F1", "F2", "F3"])
    ini = t[t["inicial"]].iloc[0]
    linhas[NOMES[m]] = {"escolhida só com F1–F3": e3["hp"],
                        "recall F4 (escolhida)": e3["recall_F4"],
                        "recall F4 (inicial)": ini["recall_F4"],
                        "diferença em F4": e3["recall_F4"] - ini["recall_F4"],
                        "é a inicial": bool(e3["inicial"])}
pd.DataFrame(linhas).T

,escolhida só com F1–F3,recall F4 (escolhida),recall F4 (inicial),diferença em F4,é a inicial
Regressão logística,"{""C"": 0.1, ""class_weight"": null}",0.566434,0.566434,0.0,True
Boosting,"{""l2_regularization"": 0.1, ""learning_rate"": 0.02, ""max_depth"": null, ""max_features"": 0...",0.574988,0.57299,0.001998,False
Random Forest,"{""max_depth"": 12, ""max_features"": 0.5, ""max_samples"": null, ""min_samples_leaf"": 50}",0.576923,0.576361,0.000562,False


Conclusão. Escolhendo só com F1–F3, a logística fica com a inicial, e o boosting e a Random Forest escolhem as mesmas configurações que com os quatro folds. No F4, que não entrou na escolha, ganham +0,0020 e +0,0006 de recall em relação à inicial (condições de pesquisa). O pequeno ganho do boosting mantém-se num período não usado para escolher, por isso não resulta apenas de se escolher e comparar nos mesmos folds. Na Random Forest, a diferença é desprezável.

## 6. Qual é o modelo probabilístico mais forte

A Parte B vai escolher um limiar sobre as **probabilidades** previstas, por isso interessa o modelo com melhor ordenação e probabilidades fiáveis. Para cada modelo na versão escolhida na secção 4 calculam-se:

- recall@200, PR-AUC normalizada e ROC-AUC: **qualidade da ordenação**;
- **Brier score** e **log-loss**: qualidade das probabilidades (menor é melhor);
- **ECE**: erro de calibração esperado, a diferença média entre a probabilidade prevista e a frequência observada de falhas (menor é melhor).

A escolha usa a regra completa da secção 8.9, porque é uma comparação entre famílias: maior recall médio, melhor em 3 de 4 folds e pior fold não inferior; em empate, menor desvio-padrão, depois maior PR-AUC normalizada e por fim o mais simples (logística, boosting, Random Forest, por custo de treino e interpretabilidade).

In [23]:
linhas = {}
for m in MODELOS:
    res, sc = final[m][versao[m]]
    linhas[NOMES[m]] = {**metricas_fold(res).to_dict(), **ot.metricas_probabilisticas(sc)}
tabela_prob = pd.DataFrame(linhas).T
tabela_prob.round(4)

,recall@200 (média),desvio,pior fold,PR-AUC norm. (média),ROC-AUC (média),Brier,log-loss,ECE
Regressão logística,0.5788,0.0136,0.5664,0.5202,0.8298,0.1044,0.3453,0.0138
Boosting,0.5877,0.0140,0.5744,0.5466,0.8358,0.1017,0.3375,0.0097
Random Forest,0.5877,0.0138,0.5759,0.5467,0.8344,0.1016,0.3368,0.0057


In [24]:
# Critério: regra completa da secção 8.9 (comparação entre famílias)
res_prob = {m: final[m][versao[m]][0] for m in MODELOS}
chave_prob, tabela_dec = ot.decidir(res_prob, simplicidade=["logistica", "boosting", "random_forest"])
MODELO_PROB = NOMES[chave_prob]
print("Modelo probabilístico mais forte:", MODELO_PROB)
print("Desvio-padrão do recall:", {NOMES[m]: round(r["recall"].std(ddof=1), 4) for m, r in res_prob.items()})
tabela_dec

Modelo probabilístico mais forte: Random Forest
Desvio-padrão do recall: {'Regressão logística': np.float64(0.0136), 'Boosting': np.float64(0.014), 'Random Forest': np.float64(0.0138)}


,diferença média,desvio da diferença,folds em que A ganha,pior fold A,pior fold B,melhoria real e estável (regra 8.9)
random_forest contra boosting,0.000013,0.001185,2,0.575858,0.574426,False
random_forest contra logistica,0.008863,0.000611,4,0.575858,0.566434,True


Conclusão. A Random Forest (configuração inicial) e o boosting otimizado têm o mesmo recall médio (0,5877), e cada um é melhor em 2 folds, por isso empatam. O primeiro desempate da secção 8.9, o menor desvio-padrão, prefere a Random Forest (0,0138 contra 0,0140). A Random Forest tem também a maior PR-AUC normalizada (0,5467 contra 0,5466) e as melhores probabilidades: Brier 0,1016, log-loss 0,3368 e ECE 0,0057, contra 0,1017, 0,3375 e 0,0097. O modelo probabilístico mais forte é a Random Forest com a configuração inicial, o mesmo modelo final da Task 3. Os três modelos têm erro de calibração baixo (ECE entre 0,006 e 0,014).

## 7. Ficheiros para a Parte B

A Parte B precisa de duas coisas: as **previsões de validação** de cada fold (cada fold previsto por um modelo ajustado só com dados anteriores, por isso o limiar pode ser escolhido nelas sem ver o teste) e os **hiperparâmetros** a usar no modelo final. É a Parte B que treina o modelo final com todo o período de desenvolvimento.

In [25]:
scores_finais = {m: final[m][versao[m]][1] for m in MODELOS}
ot.guardar_para_B(PASTA, df, config_final, scores_finais, chave_prob,
                  extra={"hiperparametros_iniciais": INICIAL, "versao_escolhida": versao,
                         "alertas_por_1000": X_ALERTAS, "amostra": AMOSTRA},
                  aquecimento_por_modelo={m: aquec[m][versao[m]] for m in MODELOS})
for f in sorted(PASTA.glob("task4_*")):
    print(f"{f.name:<45} {f.stat().st_size / 1e6:6.2f} MB")
previsoes = pd.read_csv(PASTA / f"task4_oof_validacao_{chave_prob}.csv.gz")
print(previsoes.groupby("fold").size().to_dict(), "| colunas:", list(previsoes.columns))
previsoes.head(3)
aq = pd.read_csv(PASTA / f"task4_aquecimento_{chave_prob}.csv.gz", parse_dates=["Start Time"])
aq.groupby("fold")["Start Time"].agg(["min", "max", "size"])

task4_aquecimento_boosting.csv.gz               0.29 MB
task4_aquecimento_logistica.csv.gz              0.29 MB
task4_aquecimento_random_forest.csv.gz          0.29 MB
task4_modelos_escolhidos.json                   0.00 MB
task4_oof_validacao_boosting.csv.gz             3.85 MB
task4_oof_validacao_logistica.csv.gz            3.87 MB
task4_oof_validacao_random_forest.csv.gz        3.88 MB
{'F1': 60364, 'F2': 41797, 'F3': 57542, 'F4': 75769} | colunas: ['fold', 'linha_df', 'Start Time', 'y', 'score']


,min,max,size
fold,,,
F1,2020-09-24 00:06:58,2020-09-30 23:59:20,4391
F2,2020-12-25 00:04:02,2020-12-31 23:59:38,5065
F3,2021-03-25 00:10:17,2021-03-31 23:58:56,3558
F4,2021-06-24 00:00:43,2021-06-30 23:56:09,4572


Conclusão. Ficam guardados, para os três modelos nas versões escolhidas, os hiperparâmetros (JSON), as previsões de validação dos quatro folds e os scores de aquecimento: os 7 dias antes de cada validação, previstos pelo modelo do fold, necessários para o limiar móvel da primeira semana (secção 8.8). O modelo final é a Random Forest com 200 árvores, folha mínima de 50 sessões e metade das variáveis por divisão, sem informação espacial.

**Verificação da reprodutibilidade do boosting.** Na avaliação acima, o boosting inicial difere até 0,001 por fold dos valores da pesquisa (secção 3.2), ao contrário da logística e da Random Forest. A tabela seguinte mostra que, no mesmo computador, os resultados com e sem os scores de aquecimento são iguais: a diferença vem de a pesquisa e a avaliação final terem sido corridas em computadores diferentes, e não do procedimento. Diferenças desta ordem não são distinguíveis do ruído.

In [26]:
# Verificação: o boosting inicial dá o mesmo com e sem aquecimento?
r_sem = avaliar_folds(df, ab.fabricar_scorer("boosting", GRUPOS_TODOS, ESPACO, **INICIAL["boosting"]), X_ALERTAS)
ini_csv = tabelas["boosting"][tabelas["boosting"]["inicial"].astype(bool)].iloc[0]
pd.DataFrame({"sem aquecimento": r_sem["recall"],
              "com aquecimento (secção 4)": final["boosting"]["inicial"][0]["recall"],
              "pesquisa (CSV)": [ini_csv[f"recall_{f}"] for f in ["F1", "F2", "F3", "F4"]]}).round(4)

,sem aquecimento,com aquecimento (secção 4),pesquisa (CSV)
F1,0.5973,0.5973,0.5974
F2,0.5759,0.5759,0.5751
F3,0.5999,0.5999,0.5993
F4,0.5719,0.5719,0.5730


## 8. Conclusões

**Modelos otimizados.** Random Forest, boosting e regressão logística: os dois melhores modelos da Task 3 e o melhor modelo interpretável, que serve de referência probabilística.

**Procedimento.** Os 4 folds temporais da Task 2, com o pré-processamento reajustado em cada fold só com o treino desse fold; o teste nunca foi carregado. Grelha completa na logística (16 configurações) e pesquisa aleatória com semente fixa no boosting (40 em 24 000) e na Random Forest (16 em 192). A métrica otimizada é o recall com 200 alertas por 1000 sessões, com diferenças inferiores a 0,005 desempatadas pela PR-AUC normalizada (secção 8.9).

**Inicial contra otimizado.** Só o boosting otimizado cumpre a regra completa da secção 8.9: +0,0014 de recall médio, melhor em 3 de 4 folds e pior fold mais alto (0,5744 contra 0,5719). O ganho repete-se no F4 quando a escolha é feita só com F1–F3 (+0,0020), mas é inferior ao erro de amostragem do recall num fold (0,004 a 0,006). Na logística e na Random Forest mantém-se a configuração inicial. O desempenho está limitado pela informação disponível, e não pela configuração dos modelos.

**Modelo probabilístico mais forte.** Random Forest com a configuração inicial. Empata com o boosting otimizado no recall (0,5877) e é preferida pelo menor desvio-padrão entre folds (0,0138 contra 0,0140). Tem também a maior PR-AUC normalizada (0,5467) e as probabilidades mais bem calibradas (Brier 0,1016, ECE 0,0057).

**Para a Parte B.** Ficam em `Resultados/task4_*`, para os três modelos nas versões escolhidas, três ficheiros:
- os hiperparâmetros (`task4_modelos_escolhidos.json`);
- as previsões de validação dos quatro folds (`task4_oof_validacao_<modelo>.csv.gz`);
- os scores de aquecimento, isto é, os 7 dias antes de cada validação previstos pelo modelo do fold (`task4_aquecimento_<modelo>.csv.gz`).

O modelo final é a Random Forest com 200 árvores, folha mínima de 50 sessões e metade das variáveis por divisão, sem informação espacial.